# Tutorial: Arabic Sentence Chunking

This notebook demonstrates how to chunk Arabic text into sentences. We will cover two approaches:
1. **General Approach**: Using `nltk` for standard Arabic text.
2. **Specific Approach**: Using Regular Expressions to handle Quranic waqf (stop) marks as sentence boundaries.

Audience:
- Developers working with Arabic NLP or Quranic text analysis.

Prerequisites:
- Basic Python knowledge.
- `nltk` and `re` libraries.

Learning goals:
- Understand how to tokenize Arabic sentences using standard libraries.
- Learn how to handle specialized text like the Quran using regex markers.

## Setup

We'll start by importing the necessary libraries. If you don't have `nltk` installed, you can install it via `pip install nltk`.

In [1]:
import re
import nltk

# Download necessary tokenizer data
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Setup complete.")

Setup complete.


## Fetching Text from API

Instead of hardcoding the text, we will fetch it from the **Al-Quran Cloud API**. We will use Ayat al-Kursi (2:255), which corresponds to ayah index 262 in the global count.

In [2]:
import requests

ayah_url = "https://api.alquran.cloud/v1/ayah/262"
response = requests.get(ayah_url)
ayah_data = response.json()

if response.status_code == 200:
    ayah_info = ayah_data["data"]
    text = ayah_info["text"]
    hizb_quarter = ayah_info["hizbQuarter"]
    print(f"Fetched Ayah {ayah_info['number']} (Surah {ayah_info['surah']['name']})")
    print(f"Hizb Quarter: {hizb_quarter}")
else:
    print(f"Failed to fetch ayah. Status code: {response.status_code}")

Fetched Ayah 262 (Surah سُورَةُ البَقَرَةِ)
Hizb Quarter: 17


## Fetching Context for LLM

To provide context for an LLM, we fetch the entire text of the corresponding `hizbQuarter`.

In [3]:
hizb_url = f"https://api.alquran.cloud/v1/hizbQuarter/{hizb_quarter}/quran-uthmani"
hizb_response = requests.get(hizb_url)
hizb_data = hizb_response.json()

if hizb_response.status_code == 200:
    # Combine all ayahs in the hizbQuarter for context
    context_text = " ".join([a["text"] for a in hizb_data["data"]["ayahs"]])
    print(f"Fetched Context (Hizb Quarter {hizb_quarter}) - Total Length: {len(context_text)} characters")
else:
    context_text = ""
    print("Failed to fetch context.")

Fetched Context (Hizb Quarter 17) - Total Length: 3208 characters


## Method 1: Using NLTK (General Approach)

`nltk.sent_tokenize` is great for standard prose but might struggle with Quranic text if common punctuation (like periods) is missing.

In [4]:
nltk_sentences = nltk.sent_tokenize(text)

print(f"NLTK found {len(nltk_sentences)} sentence(s):")
for i, sent in enumerate(nltk_sentences, 1):
    print(f"{i}. {sent.strip()}\n")

NLTK found 1 sentence(s):
1. ٱللَّهُ لَاۤ إِلَـٰهَ إِلَّا هُوَ ٱلۡحَیُّ ٱلۡقَیُّومُۚ لَا تَأۡخُذُهُۥ سِنَةࣱ وَلَا نَوۡمࣱۚ لَّهُۥ مَا فِی ٱلسَّمَـٰوَ ٰ⁠تِ وَمَا فِی ٱلۡأَرۡضِۗ مَن ذَا ٱلَّذِی یَشۡفَعُ عِندَهُۥۤ إِلَّا بِإِذۡنِهِۦۚ یَعۡلَمُ مَا بَیۡنَ أَیۡدِیهِمۡ وَمَا خَلۡفَهُمۡۖ وَلَا یُحِیطُونَ بِشَیۡءࣲ مِّنۡ عِلۡمِهِۦۤ إِلَّا بِمَا شَاۤءَۚ وَسِعَ كُرۡسِیُّهُ ٱلسَّمَـٰوَ ٰ⁠تِ وَٱلۡأَرۡضَۖ وَلَا یَـُٔودُهُۥ حِفۡظُهُمَاۚ وَهُوَ ٱلۡعَلِیُّ ٱلۡعَظِیمُ



## Method 2: Regex for Quranic Waqf Marks (Specific Approach)

In the Quran, waqf marks like ۚ (Jim), ۗ (Qalat), and ۖ (Salat) are the actual indicators for pauses or sentence endings. We can use regex to split the text based on these characters.

In [5]:
# Define the waqf marks as split patterns
# \u06d6 = ۖ (Salat)
# \u06d7 = ۗ (Qalat)
# \u06da = ۚ (Jim)

waqf_pattern = r"([ۚۗۖ])"

# Split while keeping the delimiter
parts = re.split(waqf_pattern, text)

# Reconstruct sentences by joining the mark back to the preceding text
regex_sentences = []
for i in range(0, len(parts)-1, 2):
    regex_sentences.append(parts[i] + parts[i+1])

# Add the last part if it exists
if len(parts) % 2 != 0 and parts[-1].strip():
    regex_sentences.append(parts[-1])

print(f"Regex found {len(regex_sentences)} sentence(s) based on waqf marks:")
for i, sent in enumerate(regex_sentences, 1):
    print(f"{i}. {sent.strip()}\n")

Regex found 9 sentence(s) based on waqf marks:
1. ٱللَّهُ لَاۤ إِلَـٰهَ إِلَّا هُوَ ٱلۡحَیُّ ٱلۡقَیُّومُۚ

2. لَا تَأۡخُذُهُۥ سِنَةࣱ وَلَا نَوۡمࣱۚ

3. لَّهُۥ مَا فِی ٱلسَّمَـٰوَ ٰ⁠تِ وَمَا فِی ٱلۡأَرۡضِۗ

4. مَن ذَا ٱلَّذِی یَشۡفَعُ عِندَهُۥۤ إِلَّا بِإِذۡنِهِۦۚ

5. یَعۡلَمُ مَا بَیۡنَ أَیۡدِیهِمۡ وَمَا خَلۡفَهُمۡۖ

6. وَلَا یُحِیطُونَ بِشَیۡءࣲ مِّنۡ عِلۡمِهِۦۤ إِلَّا بِمَا شَاۤءَۚ

7. وَسِعَ كُرۡسِیُّهُ ٱلسَّمَـٰوَ ٰ⁠تِ وَٱلۡأَرۡضَۖ

8. وَلَا یَـُٔودُهُۥ حِفۡظُهُمَاۚ

9. وَهُوَ ٱلۡعَلِیُّ ٱلۡعَظِیمُ



## Preparing Final Records (DB & LLM Ready)

We now combine the chunked sentences with metadata and context into a final JSON structure.

In [6]:
final_records = []

for i, sentence in enumerate(regex_sentences, 1):
    record = {
        "id": f"ayah_262_sent_{i}",
        "text": sentence.strip(),
        "metadata": {
            "ayah_number": ayah_info["number"],
            "surah": ayah_info["surah"],
            "number_in_surah": ayah_info["numberInSurah"],
            "juz": ayah_info["juz"],
            "hizb_quarter": ayah_info["hizbQuarter"],
            "page": ayah_info["page"],
            "source_url": ayah_url
        },
        "llm_context": context_text,
        "chunk_index": i
    }
    final_records.append(record)

print(f"Prepared {len(final_records)} records.")
# Display the first record as a sample
import json
print(json.dumps(final_records[0], indent=2, ensure_ascii=False))

Prepared 9 records.
{
  "id": "ayah_262_sent_1",
  "text": "ٱللَّهُ لَاۤ إِلَـٰهَ إِلَّا هُوَ ٱلۡحَیُّ ٱلۡقَیُّومُۚ",
  "metadata": {
    "ayah_number": 262,
    "surah": {
      "number": 2,
      "name": "سُورَةُ البَقَرَةِ",
      "englishName": "Al-Baqara",
      "englishNameTranslation": "The Cow",
      "numberOfAyahs": 286,
      "revelationType": "Medinan"
    },
    "number_in_surah": 255,
    "juz": 3,
    "hizb_quarter": 17,
    "page": 42,
    "source_url": "https://api.alquran.cloud/v1/ayah/262"
  },
  "llm_context": "۞ تِلْكَ ٱلرُّسُلُ فَضَّلْنَا بَعْضَهُمْ عَلَىٰ بَعْضٍۢ ۘ مِّنْهُم مَّن كَلَّمَ ٱللَّهُ ۖ وَرَفَعَ بَعْضَهُمْ دَرَجَٰتٍۢ ۚ وَءَاتَيْنَا عِيسَى ٱبْنَ مَرْيَمَ ٱلْبَيِّنَٰتِ وَأَيَّدْنَٰهُ بِرُوحِ ٱلْقُدُسِ ۗ وَلَوْ شَآءَ ٱللَّهُ مَا ٱقْتَتَلَ ٱلَّذِينَ مِنۢ بَعْدِهِم مِّنۢ بَعْدِ مَا جَآءَتْهُمُ ٱلْبَيِّنَٰتُ وَلَٰكِنِ ٱخْتَلَفُوا۟ فَمِنْهُم مَّنْ ءَامَنَ وَمِنْهُم مَّن كَفَرَ ۚ وَلَوْ شَآءَ ٱللَّهُ مَا ٱقْتَتَلُوا۟ وَلَٰكِنَّ ٱللَّهَ يَفْعَلُ مَا يُرِيدُ ي

## Conclusion

As seen above, `nltk` might treat the entire verse as a single sentence if it doesn't recognize the waqf marks. The **Regex approach** is much more effective for religious or specialized Arabic texts.